[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/24_ensemble_learning/90_exercises.ipynb)

# Bài tập — Chương 24: Ensemble Learning (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/24_ensemble_learning

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize
from scipy.stats import binom
from sklearn.base import clone
from sklearn.datasets import load_digits, make_friedman1
from sklearn.ensemble import (ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor,
                              StackingRegressor, VotingClassifier)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true

warnings.filterwarnings("ignore")
rng = np.random.default_rng(0)


def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


# Dữ liệu hồi quy phi tuyến dùng cho Bài 6–10
X_f, y_f = make_friedman1(3000, noise=1.0, random_state=0)
Xf_tmp, Xf_te, yf_tmp, yf_te = train_test_split(X_f, y_f, test_size=0.4, random_state=0)
Xf_tr, Xf_val, yf_tr, yf_val = train_test_split(Xf_tmp, yf_tmp, test_size=0.3, random_state=1)

## L0 — Nhớ khái niệm

**Câu 1.** (a) Phát biểu phân rã ambiguity cho ensemble trung bình. (b) Hard voting và soft voting khác nhau thế nào? (c) Quy trình stacking chuẩn (3 bước). (d) Thuật toán ensemble selection của Caruana.

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Một đội có Random Forest đạt accuracy 0.91. Họ thêm 9 Random Forest nữa (khác `random_state`) và bỏ phiếu đa số, kỳ vọng "10 model độc lập p = 0.91 → gần như 100%". Kết quả chỉ 0.912. Giải thích, và đề xuất cách tạo ensemble có lợi hơn.

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** (a) 3 classifier độc lập, mỗi cái đúng với xác suất 0.7. Xác suất bỏ phiếu đa số đúng? (b) Với 5 classifier như vậy? (c) 3 classifier độc lập với accuracy 0.9, 0.6, 0.6: xác suất đa số đúng? So với model tốt nhất?

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
p_maj3 = None
p_maj5 = None
p_mixed = None
### END CODE HERE ###
print(f"(a) {p_maj3:.4f} | (b) {p_maj5:.4f} | (c) {p_mixed:.4f} (model tốt nhất: 0.9)")

In [ ]:
check_close(p_maj3, 0.784, message="3 model p = 0.7")
check_close(p_maj5, 0.83692, tol=1e-5, message="5 model p = 0.7")
check_close(p_mixed, 0.792, message="0.9 + 0.6 + 0.6")
check_less(p_mixed, 0.9, "hai model yếu kéo đa số xuống dưới model mạnh")

**Câu 4.** Ba model dự đoán giá một căn nhà: 3, 5, 7 (tỉ VND); giá thật 4. Tính lỗi bình phương trung bình của các model, độ đa dạng (ambiguity), lỗi của ensemble trung bình; kiểm tra phân rã.

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
avg_member_err4 = None
ambiguity4 = None
ensemble_err4 = None
### END CODE HERE ###
print(f"lỗi TB thành viên = {avg_member_err4:.4f} | ambiguity = {ambiguity4:.4f} | lỗi ensemble = {ensemble_err4}")

In [ ]:
check_close(avg_member_err4, 11 / 3, message="lỗi trung bình thành viên")
check_close(ambiguity4, 8 / 3, message="độ đa dạng")
check_close(ensemble_err4, avg_member_err4 - ambiguity4, message="phân rã ambiguity")

## L3 — Code

**Bài 5.** Cài đặt `hard_vote(preds, n_classes)` (preds: mảng (n_models, n_samples) nhãn; hoà phiếu → nhãn nhỏ nhất) và `soft_vote(probas, weights=None)` (probas: (n_models, n_samples, n_classes)). So với `VotingClassifier` trên digits.

In [ ]:
def hard_vote(preds, n_classes):
    ### START CODE HERE ### (≈ 2 dòng)
    return None   # đếm phiếu từng lớp cho mỗi mẫu (np.apply_along_axis + np.bincount hoặc vòng lặp), argmax
    ### END CODE HERE ###


def soft_vote(probas, weights=None):
    ### START CODE HERE ### (≈ 2 dòng)
    return None   # trung bình (có trọng số) theo trục model rồi argmax
    ### END CODE HERE ###

In [ ]:
Xd, yd = load_digits(return_X_y=True)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.4, random_state=0, stratify=yd)
members5 = [("lr", make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))),
            ("knn", make_pipeline(StandardScaler(), KNeighborsClassifier(5))),
            ("rf", RandomForestClassifier(200, random_state=0)), ("nb", GaussianNB())]
fitted = [clone(m).fit(Xd_tr, yd_tr) for _, m in members5]
preds5 = np.array([m.predict(Xd_te) for m in fitted])
probas5 = np.array([m.predict_proba(Xd_te) for m in fitted])
w5 = [1, 2, 1, 0.5]
for voting, mine in [("hard", hard_vote(preds5, 10)), ("soft", soft_vote(probas5))]:
    ref = VotingClassifier(members5, voting=voting).fit(Xd_tr, yd_tr).predict(Xd_te)
    check_true(np.array_equal(mine, ref), f"{voting} voting khớp sklearn")
ref_w = VotingClassifier(members5, voting="soft", weights=w5).fit(Xd_tr, yd_tr).predict(Xd_te)
check_true(np.array_equal(soft_vote(probas5, w5), ref_w), "soft voting có trọng số khớp sklearn")
check_equal(hard_vote(np.array([[0], [1]]), 3)[0], 0, "hoà phiếu → nhãn nhỏ nhất")

**Bài 6.** Cài đặt `StackingScratch(base_models, meta_model, cv)` cho hồi quy:
- `fit`: với mỗi base model, tạo dự đoán out-of-fold bằng vòng lặp `cv.split` (dùng `clone`); fit meta-model trên ma trận OOF; fit lại mỗi base model trên toàn bộ dữ liệu.
- `predict`: base model dự đoán → meta-model kết hợp.

So với `StackingRegressor` cùng `cv` và cùng meta-model — dự đoán phải trùng khớp.

In [ ]:
class StackingScratch:
    def __init__(self, base_models, meta_model, cv):
        self.base_models, self.meta_model, self.cv = base_models, meta_model, cv

    def fit(self, X, y):
        ### START CODE HERE ### (≈ 9 dòng)
        # oof = zeros((n, số base model)); với mỗi (j, model) và mỗi (tr, va) trong cv.split(X): oof[va, j] = clone(model).fit(X[tr], y[tr]).predict(X[va])
        # self.meta_ = clone(meta_model).fit(oof, y); self.bases_ = [clone(m).fit(X, y) ...]
        pass
        ### END CODE HERE ###
        return self

    def predict(self, X):
        ### START CODE HERE ### (≈ 2 dòng)
        return None
        ### END CODE HERE ###

In [ ]:
bases6 = [make_pipeline(PolynomialFeatures(2), StandardScaler(), LinearRegression()),
          make_pipeline(StandardScaler(), KNeighborsRegressor(10)),
          make_pipeline(StandardScaler(), SVR(C=10)),
          HistGradientBoostingRegressor(random_state=0)]
cv6 = KFold(5, shuffle=True, random_state=0)
mine6 = StackingScratch(bases6, LinearRegression(positive=True), cv6).fit(Xf_tr, yf_tr)
ref6 = StackingRegressor([(f"m{j}", m) for j, m in enumerate(bases6)], final_estimator=LinearRegression(positive=True), cv=cv6).fit(Xf_tr, yf_tr)
p_mine6, p_ref6 = mine6.predict(Xf_te), ref6.predict(Xf_te)
print(f"RMSE test: scratch = {rmse(yf_te, p_mine6):.4f} | sklearn = {rmse(yf_te, p_ref6):.4f} | trọng số meta = {mine6.meta_.coef_.round(3)}")
check_close(p_mine6, p_ref6, tol=1e-6, message="dự đoán khớp StackingRegressor")
check_close(mine6.oof_[:, 0], cross_val_predict(bases6[0], Xf_tr, yf_tr, cv=cv6), tol=1e-8, message="OOF khớp cross_val_predict")
check_less(rmse(yf_te, p_mine6), min(rmse(yf_te, clone(m).fit(Xf_tr, yf_tr).predict(Xf_te)) for m in bases6), "stacking tốt hơn mọi base model")

## L4 — Debug

**Bài 7.** Đoạn stacking dưới đây có lỗi: meta-model học trên dự đoán **in-sample** (base model dự đoán lại chính dữ liệu đã train) → dồn trọng số cho cây quyết định overfit. Sửa (dùng OOF 5-fold `KFold(5, shuffle=True, random_state=0)`), lưu `weights7` (trọng số meta-model theo thứ tự `bases7`) và `rmse7` (RMSE test).

In [ ]:
bases7 = {"poly2": make_pipeline(PolynomialFeatures(2), StandardScaler(), LinearRegression()),
          "knn": make_pipeline(StandardScaler(), KNeighborsRegressor(10)),
          "tree": DecisionTreeRegressor(random_state=0),
          "svr": make_pipeline(StandardScaler(), SVR(C=10))}
### START CODE HERE ### (≈ 6 dòng)
Z_tr = np.column_stack([m.fit(Xf_tr, yf_tr).predict(Xf_tr) for m in bases7.values()])   # 🐛 in-sample
meta7 = LinearRegression(positive=True).fit(Z_tr, yf_tr)
Z_te = np.column_stack([m.predict(Xf_te) for m in bases7.values()])
weights7, rmse7 = meta7.coef_, rmse(yf_te, meta7.predict(Z_te))
### END CODE HERE ###
print("trọng số:", dict(zip(bases7, np.round(weights7, 3))), f"| RMSE test = {rmse7:.4f}")

In [ ]:
best_single7 = min(rmse(yf_te, clone(m).fit(Xf_tr, yf_tr).predict(Xf_te)) for m in bases7.values())
check_less(weights7[list(bases7).index("tree")], 0.2, "cây overfit không còn chiếm trọng số")
check_less(rmse7, best_single7, "stacking OOF tốt hơn base model tốt nhất")

## L5 — Phân tích

**Bài 8.** So sánh hai ensemble trung bình 5 thành viên trên dữ liệu Friedman (train `Xf_tr`, đánh giá `Xf_te`):
- **Cùng loại**: 5 `RandomForestRegressor(200)` với `random_state` 0..4.
- **Đa dạng**: SVR(C=10) (đã chuẩn hoá), RandomForest(200, seed 0), HistGB, KNN(10) (đã chuẩn hoá), hồi quy đa thức bậc 2.

Với mỗi ensemble tính: `corr_*` = tương quan trung bình giữa các cặp vector phần dư; `gain_*` = (MSE trung bình thành viên − MSE ensemble) / MSE trung bình thành viên; `ens_mse_*`, `best_member_mse_*`. Nhận xét.

In [ ]:
def ensemble_stats(models):
    ### START CODE HERE ### (≈ 7 dòng)
    # P = dự đoán test của từng model (cột); R = P − y; mse_members = (R**2).mean(0); ens_mse = MSE của P.mean(1)
    # corr = trung bình các phần tử ngoài đường chéo của np.corrcoef(R.T)
    # gain = (MSE trung bình thành viên − ens_mse) / MSE trung bình thành viên
    corr, gain, ens_mse, best_member_mse = None, None, None, None
    return corr, gain, ens_mse, best_member_mse
    ### END CODE HERE ###


same8 = [RandomForestRegressor(200, random_state=s, n_jobs=-1) for s in range(5)]
diverse8 = [make_pipeline(StandardScaler(), SVR(C=10)), RandomForestRegressor(200, random_state=0, n_jobs=-1),
            HistGradientBoostingRegressor(random_state=0), make_pipeline(StandardScaler(), KNeighborsRegressor(10)),
            make_pipeline(PolynomialFeatures(2), StandardScaler(), LinearRegression())]
corr_same, gain_same, ens_mse_same, best_member_mse_same = ensemble_stats(same8)
corr_div, gain_div, ens_mse_div, best_member_mse_div = ensemble_stats(diverse8)
print(f"cùng loại: tương quan phần dư = {corr_same:.3f} | lợi ích = {gain_same:.1%} | MSE ensemble = {ens_mse_same:.3f} (thành viên tốt nhất {best_member_mse_same:.3f})")
print(f"đa dạng  : tương quan phần dư = {corr_div:.3f} | lợi ích = {gain_div:.1%} | MSE ensemble = {ens_mse_div:.3f} (thành viên tốt nhất {best_member_mse_div:.3f})")

In [ ]:
check_greater(corr_same, 0.95, "các rừng khác seed gần như trùng lỗi")
check_less(corr_div, corr_same - 0.2, "ensemble đa dạng có lỗi ít tương quan hơn")
check_less(gain_same, 0.02, "lợi ích gần như bằng 0")
check_greater(gain_div, 0.2, "lợi ích lớn so với thành viên trung bình")

## L6 — Thực tế

**Bài 9.** **Ensemble dưới ràng buộc chi phí.** Hệ thống định giá cần trả lời trong thời gian giới hạn. Mỗi model có chi phí dự đoán (đơn vị tương đối) trong `COST`. Cài đặt `budgeted_selection(V, y_val, names, costs, budget, n_steps=25)`: như ensemble selection (chọn có hoàn lại theo RMSE validation), nhưng **không được** thêm model mới nếu tổng chi phí các model **khác nhau** đã dùng vượt `budget` (thêm lại model đã có thì không tốn thêm). Chạy với `budget=None` (không giới hạn) và `budget=6`. Lưu `rmse_free9`, `cost_free9`, `rmse_budget9`, `cost_budget9` (RMSE trên **test**), `rmse_single9` (model đơn tốt nhất theo validation, đo trên test).

In [ ]:
library9 = {"poly2": make_pipeline(PolynomialFeatures(2), StandardScaler(), LinearRegression()),
            "knn": make_pipeline(StandardScaler(), KNeighborsRegressor(10)),
            "svr": make_pipeline(StandardScaler(), SVR(C=10)),
            "rf": RandomForestRegressor(300, random_state=0, n_jobs=-1),
            "et": ExtraTreesRegressor(300, random_state=0, n_jobs=-1),
            "hgb": HistGradientBoostingRegressor(random_state=0),
            "mlp": make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=2000, random_state=0))}
COST = {"poly2": 1, "knn": 3, "svr": 8, "rf": 20, "et": 20, "hgb": 4, "mlp": 2}
names9 = list(library9)
V9 = np.column_stack([library9[n].fit(Xf_tr, yf_tr).predict(Xf_val) for n in names9])
T9 = np.column_stack([library9[n].predict(Xf_te) for n in names9])
for j, n in enumerate(names9):
    print(f"{n:6s} RMSE val = {rmse(yf_val, V9[:, j]):.3f} | test = {rmse(yf_te, T9[:, j]):.3f} | chi phí = {COST[n]}")


def budgeted_selection(V, y_val, names, costs, budget=None, n_steps=25):
    counts = np.zeros(V.shape[1])
    current = np.zeros(len(y_val))
    ### START CODE HERE ### (≈ 11 dòng)
    # mỗi bước: với mỗi j, chi phí = tổng costs của {model đã dùng} ∪ {j}; nếu vượt budget → điểm = inf,
    # ngược lại điểm = RMSE validation của (current * step + V[:, j]) / (step + 1); chọn j tốt nhất, cập nhật counts, current
    pass
    ### END CODE HERE ###
    w = counts / counts.sum()
    return w, sum(costs[names[k]] for k in np.flatnonzero(w))

In [ ]:
### START CODE HERE ### (≈ 5 dòng)
w_free, cost_free9 = None, None          # budgeted_selection(..., budget=None)
w_bud, cost_budget9 = None, None         # budgeted_selection(..., budget=6)
rmse_free9, rmse_budget9 = None, None    # RMSE test của T9 @ w
best_val = None                          # chỉ số model đơn có RMSE validation nhỏ nhất
rmse_single9 = None
### END CODE HERE ###
print(f"model đơn tốt nhất (val: {names9[best_val]}): RMSE test = {rmse_single9:.3f} | chi phí = {COST[names9[best_val]]}")
print(f"không giới hạn: RMSE test = {rmse_free9:.3f} | chi phí = {cost_free9} | trọng số = {dict(zip(names9, w_free.round(2)))}")
print(f"ngân sách 6   : RMSE test = {rmse_budget9:.3f} | chi phí = {cost_budget9} | trọng số = {dict(zip(names9, w_bud.round(2)))}")

In [ ]:
check_true(cost_budget9 <= 6, "tôn trọng ngân sách")
check_less(rmse_budget9, rmse_single9 - 0.05, "ensemble rẻ vẫn tốt hơn rõ model đơn tốt nhất (theo val)")
check_less(rmse_free9, rmse_single9 - 0.05, "ensemble không giới hạn tốt hơn model đơn")

## L7 — Nâng cao

**Bài 10.** **Trọng số lồi (convex combination) bằng projected gradient descent.** Tìm $\mathbf w$ tối thiểu $\frac1n\lVert V\mathbf w - \mathbf y\rVert^2$ với ràng buộc $w_j \ge 0$, $\sum_j w_j = 1$ (simplex).
1. Cài đặt `project_simplex(v)` — phép chiếu Euclid lên simplex (Duchi et al., 2008): sắp xếp giảm dần $u$, tìm $\rho = \max\{j: u_j - \frac1j(\sum_{i \le j}u_i - 1) > 0\}$, $\theta = \frac1\rho(\sum_{i \le \rho}u_i - 1)$, trả về $\max(v - \theta, 0)$.
2. `convex_weights(V, y, n_iter)`: projected gradient với bước $1/L$, $L = 2\lambda_{\max}(G)$, $G = V^\top V/n$. Tính trước $G$ và $\mathbf b = V^\top\mathbf y/n$ → gradient $2(G\mathbf w - \mathbf b)$ chỉ tốn phép nhân ma trận $M \times M$ (các cột của $V$ tương quan rất mạnh → bài toán điều kiện xấu, cần nhiều vòng lặp). So với SLSQP của scipy (`w_slsqp`).
3. **Khi nào ràng buộc có lợi?** Với validation nhỏ $n_{val} \in \{30, 150\}$ (30 tập con ngẫu nhiên của validation, seed 0..29), so RMSE test trung bình của trọng số simplex với trọng số tự do (`np.linalg.lstsq`): lưu `small_val[n_val] = (rmse_simplex, rmse_free)`.

In [ ]:
def project_simplex(v):
    ### START CODE HERE ### (≈ 5 dòng)
    return None
    ### END CODE HERE ###


def convex_weights(V, y, n_iter=20000):
    w = np.full(V.shape[1], 1 / V.shape[1])
    ### START CODE HERE ### (≈ 5 dòng)
    # G = VᵀV / n; b = Vᵀy / n; L = 2 * λ_max(G)
    # lặp: grad = 2 (G w − b); w = project_simplex(w − grad / L)
    pass
    ### END CODE HERE ###
    return w

In [ ]:
check_close(project_simplex(np.array([0.2, 0.3, 0.5])), [0.2, 0.3, 0.5], message="điểm đã trên simplex giữ nguyên")
check_close(project_simplex(np.array([2.0, 0.0, 0.0])), [1.0, 0.0, 0.0], message="chiếu về đỉnh")
check_close(project_simplex(np.array([0.5, 0.5, 0.5])), [1 / 3] * 3, message="chiếu đối xứng")

w_pgd = convex_weights(V9, yf_val)
res10 = minimize(lambda w: np.mean((V9 @ w - yf_val) ** 2), np.full(len(names9), 1 / len(names9)), method="SLSQP",
                 bounds=[(0, 1)] * len(names9), constraints={"type": "eq", "fun": lambda w: w.sum() - 1})
w_slsqp = res10.x
print("PGD  :", dict(zip(names9, w_pgd.round(3))), f"| RMSE test = {rmse(yf_te, T9 @ w_pgd):.4f}")
print("SLSQP:", dict(zip(names9, w_slsqp.round(3))))
check_close(w_pgd.sum(), 1.0, message="tổng trọng số = 1")
check_true((w_pgd >= 0).all(), "trọng số không âm")
mse10 = lambda w: np.mean((V9 @ w - yf_val) ** 2)
check_less(mse10(w_pgd), mse10(w_slsqp) + 1e-6, "MSE validation không tệ hơn SLSQP")
check_close(w_pgd, w_slsqp, tol=0.03, message="trọng số gần SLSQP (nghiệm gần như không duy nhất khi các cột tương quan mạnh)")

In [ ]:
small_val = {}
### START CODE HERE ### (≈ 9 dòng)
# với n_val trong [30, 150]: 30 lần chọn ngẫu nhiên n_val điểm của validation (np.random.default_rng(seed).choice(..., replace=False))
# → trọng số simplex (convex_weights) và tự do (lstsq) trên tập con → RMSE test (T9 @ w); lưu trung bình
pass
### END CODE HERE ###
for n_val, (rs, rf) in small_val.items():
    print(f"n_val = {n_val:3d}: RMSE test TB — simplex = {rs:.3f} | tự do = {rf:.3f}")

In [ ]:
check_less(small_val[30][0], small_val[30][1] - 0.03, "validation nhỏ: ràng buộc simplex tốt hơn rõ")
check_greater(small_val[30][1] - small_val[30][0], small_val[150][1] - small_val[150][0], "lợi thế của ràng buộc giảm khi validation lớn")